# Singular Value Decomposition (SVD) Review

## Learning Objectives

1. **State** the SVD theorem and understand its three matrix components
2. **Implement** a simplified SVD via power iteration
3. **Visualize** the geometric meaning of the decomposition
4. **State** the Eckart-Young theorem for best low-rank approximation

## The SVD Theorem

Every real matrix $A \in \mathbb{R}^{m \times n}$ can be written as:

$$A = U \Sigma V^\top$$

where:
- $U \in \mathbb{R}^{m \times m}$: orthonormal columns (left singular vectors)
- $\Sigma \in \mathbb{R}^{m \times n}$: diagonal, $\sigma_1 \geq \sigma_2 \geq \ldots \geq 0$ (singular values)
- $V \in \mathbb{R}^{n \times n}$: orthonormal columns (right singular vectors)

**Equivalent form:** $A = \sum_{i=1}^{r} \sigma_i u_i v_i^\top$ where $r = \text{rank}(A)$.

Each term is a **rank-1 matrix** weighted by $\sigma_i$.

## Geometric Interpretation

$A$ maps $v_i$ (unit vectors in input space) to $\sigma_i u_i$ (scaled vectors in output space).

1. $V^\top$ rotates the input to align with principal axes
2. $\Sigma$ scales along each axis
3. $U$ rotates into the output coordinate system

**The singular values $\sigma_i$** measure how much the matrix "stretches" in each direction.
Large $\sigma_1 \gg \sigma_2$ → data lives near a low-dimensional subspace.

**Connection to eigendecomposition:**
- $A^\top A = V \Sigma^\top U^\top U \Sigma V^\top = V (\Sigma^\top \Sigma) V^\top$
- Eigenvalues of $A^\top A$ are $\sigma_i^2$; eigenvectors are $v_i$ (right singular vectors)

In [ ]:
import math

def mat_mult(A, B):
    m,k = len(A),len(A[0]); n=len(B[0])
    C = [[sum(A[i][l]*B[l][j] for l in range(k)) for j in range(n)] for i in range(m)]
    return C

def transpose(A):
    return [[A[j][i] for j in range(len(A))] for i in range(len(A[0]))]

def norm(v):
    return math.sqrt(sum(x**2 for x in v))

def normalize(v):
    n=norm(v); return [x/n for x in v] if n>0 else v

def mat_vec(A, v):
    return [sum(A[i][j]*v[j] for j in range(len(v))) for i in range(len(A))]

def power_iteration(A, n_iter=100):
    """Find largest singular value and vectors via power iteration."""
    m, n = len(A), len(A[0])
    import random; rng=random.Random(0)
    v = normalize([rng.gauss(0,1) for _ in range(n)])
    At = transpose(A)
    for _ in range(n_iter):
        u = normalize(mat_vec(A, v))
        v = normalize(mat_vec(At, u))
    sigma = sum(u[i] * sum(A[i][j]*v[j] for j in range(n)) for i in range(m))
    return sigma, u, v

# Small example matrix
A = [[3, 1, 1],
     [2, 4, 1],
     [0, 1, 2]]

sigma1, u1, v1 = power_iteration(A)
print(f"Largest singular value σ₁ = {sigma1:.4f}")
print(f"Left singular vector u₁  = {[round(x,4) for x in u1]}")
print(f"Right singular vector v₁ = {[round(x,4) for x in v1]}")

# Verify: A*v1 ≈ σ1*u1
Av1 = mat_vec(A, v1)
diff = [Av1[i] - sigma1*u1[i] for i in range(len(u1))]
print(f"
||A*v₁ - σ₁*u₁|| = {norm(diff):.6f}  (should be ≈ 0)")

## Eckart-Young Theorem

The best rank-$k$ approximation of $A$ in Frobenius norm is:

$$A_k = \sum_{i=1}^{k} \sigma_i u_i v_i^\top$$

$$\|A - A_k\|_F^2 = \sum_{i=k+1}^{r} \sigma_i^2$$

**Energy captured by top-$k$:**
$$\frac{\sum_{i=1}^k \sigma_i^2}{\sum_{i=1}^r \sigma_i^2}$$

This is the **fraction of variance explained** by the rank-$k$ approximation.

In [ ]:
# Compute rank-1 approximation and check residual
def rank1_approx(sigma, u, v):
    return [[sigma*u[i]*v[j] for j in range(len(v))] for i in range(len(u))]

def frob_norm_sq(M):
    return sum(M[i][j]**2 for i in range(len(M)) for j in range(len(M[0])))

# Rank-1 approximation
A1 = rank1_approx(sigma1, u1, v1)
residual = [[A[i][j]-A1[i][j] for j in range(len(A[0]))] for i in range(len(A))]

fA  = frob_norm_sq(A)
fA1 = frob_norm_sq(A1)
fR  = frob_norm_sq(residual)

print(f"||A||²_F = {fA:.4f}")
print(f"||A₁||²_F = {fA1:.4f}  (= σ₁² = {sigma1**2:.4f})")
print(f"||A - A₁||²_F = {fR:.4f}")
print(f"Energy captured by rank-1: {fA1/fA*100:.1f}%")

# Second singular value
# Deflate A by removing the rank-1 component
sigma2, u2, v2 = power_iteration(residual)
A2 = rank1_approx(sigma2, u2, v2)
residual2 = [[residual[i][j]-A2[i][j] for j in range(len(A[0]))] for i in range(len(A))]
energy_2 = (sigma1**2 + sigma2**2)/fA
print(f"
σ₂ = {sigma2:.4f}")
print(f"Energy captured by rank-2: {energy_2*100:.1f}%")
